# 08. UN Comtrade — flow1 수집 (규제국 → 중간국 수출)

## 목적
규제국이 중간국으로 수출한 데이터를 수집해 삼각무역 우회 경로를 교차검증한다.

- **flow1**: 규제국 → 중간국 (UN Comtrade 수출 기준)

노트북 07 분석에서 flow2(중간국→한국)가 증가한 국가들이  
규제 이후 규제국으로부터 수입도 늘었는지 확인한다.

## 입력
- `data/interim/regulation_events_atomic(~2015).csv` — 규제 이벤트(HS코드·기간·규제국)
- `data/interim/customs_flow0_flow2_raw.csv` — flow2 후보국 선정용

## 출력
- `data/interim/comtrade_flow1_raw.csv` — 규제국→중간국 월별 수출 데이터

## 주의
- UN Comtrade 무료 키 할당량: **500 호출/일**. 수집 중단 시 재실행하면 캐시에서 재개된다.
- Comtrade의 reporter/partner 코드는 ISO 코드와 다를 수 있다 (대만 TWN → 490 등).

## 0. 라이브러리 & 설정

In [32]:
import ast
import time
from pathlib import Path

import comtradeapicall
import pandas as pd

from src.config import get_comtrade_api_key

In [33]:
# --- 경로 설정 ---
DATA_DIR = Path("../data/interim")
INPUT_ATOMIC = DATA_DIR / "regulation_events_atomic(~2015).csv"
INPUT_FLOW2  = DATA_DIR / "customs_flow0_flow2_raw.csv"
OUTPUT_PATH  = DATA_DIR / "comtrade_flow1_raw.csv"
STATUS_PATH  = DATA_DIR / "comtrade_flow1_status.csv"

# 캐시: 호출 단위(regulated_iso3, hs_code, window, intermediary_iso2)로 parquet 저장
CACHE_DIR = DATA_DIR / "cache" / "comtrade"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# --- Comtrade API ---
API_KEY = get_comtrade_api_key()

# --- 수집 파라미터 ---
TOP_N_INTERMEDIARIES = 5   # 이벤트당 상위 중간국 후보 수 (API 호출량 제어)
CHECKPOINT_EVERY     = 20  # N호출마다 진행 상황 저장
SLEEP_BETWEEN_CALLS  = 1.2 # 연속 호출 간 대기 시간(초)

## 1. 입력 데이터 로드

In [34]:
df_atomic = pd.read_csv(INPUT_ATOMIC)
df_flow2_all = pd.read_csv(INPUT_FLOW2)

# flow=2인 행만 사용 (중간국→한국 수입)
df_f2 = df_flow2_all[df_flow2_all["flow"] == 2].copy()

# trade_date 파싱: "2016.05" → Period
df_f2["period"] = (
    pd.to_datetime(df_f2["trade_date"].astype(str), format="%Y.%m")
    .dt.to_period("M")
)

print("atomic:", df_atomic.shape)
print("flow2 :", df_f2.shape)
print("\n규제국 목록:", sorted(df_atomic["origin_country_iso3"].unique()))

atomic: (155, 20)
flow2 : (123928, 18)

규제국 목록: ['ARE', 'AUS', 'CHN', 'EGY', 'ESP', 'FIN', 'FRA', 'IDN', 'IND', 'ITA', 'JPN', 'MYS', 'SAU', 'SGP', 'THA', 'TWN', 'UKR', 'USA', 'VNM']


## 2. Comtrade 국가 코드 매핑

Comtrade API는 ISO3/ISO2가 아닌 자체 숫자 코드를 사용한다.  
`comtradeapicall.getReference()`로 최신 매핑을 가져온다.

In [35]:
# Reporter reference (수출국 = 규제국)
reporter_ref = comtradeapicall.getReference("reporter")

# 활성 국가만 (그룹 제외, 만료 제외)
reporter_active = reporter_ref[
    reporter_ref["isGroup"].eq(False) &
    reporter_ref["entryExpiredDate"].isna()
].copy()

print("reporter_ref 컬럼:", reporter_ref.columns.tolist())
reporter_ref.head(3)

reporter_ref 컬럼: ['id', 'text', 'reporterCode', 'reporterDesc', 'reporterNote', 'reporterCodeIsoAlpha2', 'reporterCodeIsoAlpha3', 'entryEffectiveDate', 'isGroup', 'entryExpiredDate']


,id,text,reporterCode,reporterDesc,reporterNote,reporterCodeIsoAlpha2,reporterCodeIsoAlpha3,entryEffectiveDate,isGroup,entryExpiredDate
0,4,Afghanistan,4,Afghanistan,Afghanistan,AF,AFG,1900-01-01T00:00:00,False,NaN
1,8,Albania,8,Albania,Albania,AL,ALB,1900-01-01T00:00:00,False,NaN
2,12,Algeria,12,Algeria,Algeria,DZ,DZA,1900-01-01T00:00:00,False,NaN


In [36]:
# ISO3 → reporter code 매핑 구성
# 컬럼명은 버전에 따라 다를 수 있으므로 후보 이름을 우선순위대로 시도
def find_col(df: pd.DataFrame, candidates: list[str]) -> str | None:
    """컬럼 후보 목록에서 실제 존재하는 첫 번째 컬럼명 반환."""
    for c in candidates:
        if c in df.columns:
            return c
    return None


reporter_iso3_col = find_col(reporter_active, [
    "ReporterCodeIsoAlpha3", "reporterCodeIsoAlpha3", "iso3", "ISO3",
])
reporter_code_col = find_col(reporter_active, [
    "ReporterCode", "reporterCode", "id", "code",
])

print(f"사용 컬럼 — ISO3: {reporter_iso3_col}, 코드: {reporter_code_col}")

# ISO3 → reporter code 딕셔너리
iso3_to_reporter = (
    reporter_active
    .dropna(subset=[reporter_iso3_col])
    .drop_duplicates(reporter_iso3_col)
    .set_index(reporter_iso3_col)[reporter_code_col]
    .astype(str)
    .to_dict()
)

# 대만(TWN)은 Comtrade에서 reporter로 직접 등록되지 않아 "490" (Other Asia, nes) 코드를 수동 매핑
# isGroup=True인 그룹 코드이므로 통계가 다른 아시아 소규모 국가를 포함할 수 있음
iso3_to_reporter["TWN"] = "490"

# 규제국 매핑 결과 확인
regulated_iso3s = sorted(df_atomic["origin_country_iso3"].unique())
print("\n규제국 Comtrade reporter 코드 매핑:")
for iso3 in regulated_iso3s:
    code = iso3_to_reporter.get(iso3, "NOT FOUND")
    print(f"  {iso3} → {code}")

사용 컬럼 — ISO3: reporterCodeIsoAlpha3, 코드: reporterCode

규제국 Comtrade reporter 코드 매핑:
  ARE → 784
  AUS → 36
  CHN → 156
  EGY → 818
  ESP → 724
  FIN → 246
  FRA → 251
  IDN → 360
  IND → 699
  ITA → 380
  JPN → 392
  MYS → 458
  SAU → 682
  SGP → 702
  THA → 764
  TWN → 490
  UKR → 804
  USA → 842
  VNM → 704


In [37]:
# Partner reference (수입국 = 중간국)
partner_ref = comtradeapicall.getReference("partner")
partner_active = partner_ref[
    partner_ref["isGroup"].eq(False) &
    partner_ref["entryExpiredDate"].isna()
].copy()

partner_iso2_col = find_col(partner_active, [
    "PartnerCodeIsoAlpha2", "partnerCodeIsoAlpha2", "iso2", "ISO2",
])
partner_code_col = find_col(partner_active, [
    "PartnerCode", "partnerCode", "id", "code",
])

print(f"사용 컬럼 — ISO2: {partner_iso2_col}, 코드: {partner_code_col}")

# ISO2 → partner code 딕셔너리
iso2_to_partner = (
    partner_active
    .dropna(subset=[partner_iso2_col])
    .drop_duplicates(partner_iso2_col)
    .set_index(partner_iso2_col)[partner_code_col]
    .astype(str)
    .to_dict()
)

print(f"매핑 가능한 partner 국가 수: {len(iso2_to_partner)}")

사용 컬럼 — ISO2: PartnerCodeIsoAlpha2, 코드: PartnerCode
매핑 가능한 partner 국가 수: 246


## 3. 중간국 후보 선정

각 규제 이벤트별로 규제 이후 한국 수입이 증가한 flow2 국가 상위 `TOP_N_INTERMEDIARIES`개를 선정한다.

In [38]:
# source_row_id → 규제 시작 period 매핑
start_map = (
    df_atomic[["source_row_id", "start_date"]]
    .drop_duplicates("source_row_id")
    .assign(start_period=lambda x: pd.to_datetime(x["start_date"]).dt.to_period("M"))
    .set_index("source_row_id")["start_period"]
)

# flow2 데이터에 규제 시작 period 추가 후 before/after 라벨링
df_f2["start_period"] = df_f2["source_row_id"].map(start_map)
df_f2["period_label"] = (df_f2["period"] >= df_f2["start_period"]).map(
    {True: "after", False: "before"}
)

# Step 1: trade month 기준 월별 합산
f2_monthly = (
    df_f2
    .groupby(
        ["source_row_id", "origin_country_iso3", "hs_code_query",
         "window_start", "window_end", "country_iso2", "period", "period_label"],
        dropna=False,
    )
    .agg(monthly_imp_dlr=("imp_dlr", "sum"))
    .reset_index()
)

# Step 2: 월평균 집계 후 pivot
f2_agg = (
    f2_monthly
    .groupby(
        ["source_row_id", "origin_country_iso3", "hs_code_query",
         "window_start", "window_end", "country_iso2", "period_label"],
        dropna=False,
    )
    .agg(avg_imp_dlr=("monthly_imp_dlr", "mean"))
    .reset_index()
    .pivot_table(
        index=["source_row_id", "origin_country_iso3", "hs_code_query",
               "window_start", "window_end", "country_iso2"],
        columns="period_label",
        values="avg_imp_dlr",
        fill_value=0.0,
    )
    .reset_index()
)

# after/before 컬럼이 없을 경우 대비
for col in ["after", "before"]:
    if col not in f2_agg.columns:
        f2_agg[col] = 0.0

f2_agg.columns.name = None
f2_agg["increase"] = f2_agg["after"] - f2_agg["before"]

print("집계 후 행 수:", len(f2_agg))
f2_agg.head()

집계 후 행 수: 8170


,source_row_id,origin_country_iso3,hs_code_query,window_start,window_end,country_iso2,after,before,increase
0,1,CHN,700529,2014-01,2016-01,AE,379453.166667,505424.818182,-125971.651515
1,1,CHN,700529,2014-01,2016-01,AO,0.000000,0.000000,0.000000
2,1,CHN,700529,2014-01,2016-01,BE,5581.000000,6541.000000,-960.000000
3,1,CHN,700529,2014-01,2016-01,BG,873110.777778,350360.000000,522750.777778
4,1,CHN,700529,2014-01,2016-01,BR,0.000000,0.000000,0.000000


In [39]:
# 규제 후 수입액 기준 상위 TOP_N_INTERMEDIARIES 선정 (after > 0 필터)
intermediary_candidates = (
    f2_agg[f2_agg["after"] > 0]
    .sort_values("after", ascending=False)
    .groupby(
        ["source_row_id", "origin_country_iso3", "hs_code_query",
         "window_start", "window_end"],
        group_keys=False,
    )
    .head(TOP_N_INTERMEDIARIES)
    [["source_row_id", "origin_country_iso3", "hs_code_query",
      "window_start", "window_end", "country_iso2", "after", "before", "increase"]]
    .reset_index(drop=True)
)

print(f"중간국 후보 조합: {len(intermediary_candidates)}건")
print(f"고유 source_row_id: {intermediary_candidates['source_row_id'].nunique()}개")
print(f"\n샘플:")
intermediary_candidates.head(10)

중간국 후보 조합: 735건
고유 source_row_id: 60개

샘플:


,source_row_id,origin_country_iso3,hs_code_query,window_start,window_end,country_iso2,after,before,increase
0,60,CHN,720851,2024-11,2026-11,JP,3.916576e+07,3.310537e+07,6.060393e+06
1,26,IND,392062,2018-09,2020-09,JP,3.059505e+07,2.866838e+07,1.926666e+06
2,26,CHN,392062,2018-09,2020-09,JP,3.059505e+07,2.866838e+07,1.926666e+06
3,30,MYS,4412,2019-11,2021-11,ID,2.647287e+07,2.247328e+07,3.999589e+06
4,32,VNM,4412,2019-11,2021-11,ID,2.647287e+07,2.247328e+07,3.999589e+06
5,31,CHN,4412,2019-11,2021-11,ID,2.647287e+07,2.247328e+07,3.999589e+06
6,37,THA,392062,2020-12,2022-12,JP,2.586520e+07,3.314202e+07,-7.276816e+06
7,37,TWN,392062,2020-12,2022-12,JP,2.586520e+07,3.314202e+07,-7.276816e+06
8,37,ARE,392062,2020-12,2022-12,JP,2.586520e+07,3.314202e+07,-7.276816e+06
9,16,TWN,392062,2017-04,2019-04,JP,2.465240e+07,2.724735e+07,-2.594942e+06


## 4. 수집 계획 생성

동일한 `(regulated_iso3, hs_code, window_start, window_end, intermediary_iso2)` 조합은  
중복 API 호출을 피하기 위해 한 번만 수집한다.

In [40]:
def make_monthly_chunks(window_start: str, window_end: str) -> list[tuple[str, str]]:
    """YYYY-MM 형식 구간을 12개월 단위 (YYYYMM, YYYYMM) 청크 리스트로 분할."""
    start = pd.Period(window_start, freq="M")
    end   = pd.Period(window_end,   freq="M")
    chunks = []
    cursor = start
    while cursor <= end:
        chunk_end = min(cursor + 11, end)
        chunks.append((cursor.strftime("%Y%m"), chunk_end.strftime("%Y%m")))
        cursor = chunk_end + 1
    return chunks


# 고유 호출 키 추출 (source_row_id 제외)
call_keys = (
    intermediary_candidates
    [["origin_country_iso3", "hs_code_query", "window_start", "window_end", "country_iso2"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Comtrade 코드 매핑
call_keys["reporter_code"] = call_keys["origin_country_iso3"].map(iso3_to_reporter)
call_keys["partner_code"]  = call_keys["country_iso2"].map(iso2_to_partner)

# 코드 매핑 실패한 행 분리
missing_codes = call_keys[
    call_keys["reporter_code"].isna() | call_keys["partner_code"].isna()
]
call_plan = call_keys.dropna(subset=["reporter_code", "partner_code"]).reset_index(drop=True)

print(f"전체 호출 조합: {len(call_keys)}건")
print(f"코드 매핑 실패: {len(missing_codes)}건")
print(f"실제 수집 대상: {len(call_plan)}건")

if not missing_codes.empty:
    print("\n[매핑 실패 목록]")
    print(missing_codes[["origin_country_iso3", "country_iso2", "reporter_code", "partner_code"]])

전체 호출 조합: 735건
코드 매핑 실패: 0건
실제 수집 대상: 735건


## 5. API 동작 테스트

본격 수집 전, 단일 건으로 응답 구조를 확인한다.

In [41]:
MAX_RETRIES = 3       # 네트워크 오류 시 최대 재시도 횟수
RETRY_SLEEP  = 10.0   # 재시도 전 대기 시간(초)


def collect_comtrade_one(
    reporter_code: str,
    partner_code: str,
    hs_code: str,
    start_yymm: str,
    end_yymm: str,
) -> pd.DataFrame:
    """
    UN Comtrade에서 수출국(reporter)→수입국(partner) 월별 수출 데이터 수집.

    None 반환이 네트워크 오류인지 할당량 소진인지 라이브러리에서 구분 불가하므로
    MAX_RETRIES회 재시도 후에도 None이면 RuntimeError를 발생시킨다.
    """
    periods = [
        p.strftime("%Y%m")
        for p in pd.period_range(start_yymm, end_yymm, freq="M")
    ]

    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            df = comtradeapicall.getFinalData(
                API_KEY,
                typeCode="C",
                freqCode="M",
                clCode="HS",
                period=",".join(periods),
                reporterCode=str(reporter_code),
                cmdCode=str(hs_code),
                flowCode="X",
                partnerCode=str(partner_code),
                partner2Code=None,
                customsCode=None,
                motCode=None,
                maxRecords=250_000,
                format_output="JSON",
                aggregateBy=None,
                breakdownMode="classic",
                countOnly=None,
                includeDesc=True,
            )
        except Exception as exc:
            last_err = exc
            df = None

        if df is not None:
            return df if not df.empty else pd.DataFrame()

        # None 반환: 재시도 또는 포기
        if attempt < MAX_RETRIES:
            print(f"  [재시도 {attempt}/{MAX_RETRIES}] {RETRY_SLEEP}초 대기 후 재시도...")
            time.sleep(RETRY_SLEEP)

    raise RuntimeError(
        f"Comtrade API가 {MAX_RETRIES}회 모두 None을 반환했습니다. "
        f"할당량 소진 또는 네트워크 오류일 수 있습니다. (마지막 오류: {last_err})"
    )

In [42]:
# 테스트: call_plan 첫 번째 행으로 응답 구조 확인
if len(call_plan) > 0:
    test_row = call_plan.iloc[0]
    test_chunks = make_monthly_chunks(test_row["window_start"], test_row["window_end"])
    test_chunk_start, test_chunk_end = test_chunks[0]  # 첫 청크만 테스트

    print(f"테스트: {test_row['origin_country_iso3']}({test_row['reporter_code']}) → "
          f"{test_row['country_iso2']}({test_row['partner_code']}), "
          f"HS {test_row['hs_code_query']}, {test_chunk_start}~{test_chunk_end}")

    df_test = collect_comtrade_one(
        reporter_code=test_row["reporter_code"],
        partner_code=test_row["partner_code"],
        hs_code=str(test_row["hs_code_query"]),
        start_yymm=test_chunk_start,
        end_yymm=test_chunk_end,
    )
    print(f"\n응답 shape: {df_test.shape}")
    if not df_test.empty:
        print("컬럼:", df_test.columns.tolist())
        display(df_test.head(5))

테스트: CHN(156) → JP(392), HS 720851, 202411~202510

응답 shape: (2, 47)
컬럼: ['typeCode', 'freqCode', 'refPeriodId', 'refYear', 'refMonth', 'period', 'reporterCode', 'reporterISO', 'reporterDesc', 'flowCode', 'flowDesc', 'partnerCode', 'partnerISO', 'partnerDesc', 'partner2Code', 'partner2ISO', 'partner2Desc', 'classificationCode', 'classificationSearchCode', 'isOriginalClassification', 'cmdCode', 'cmdDesc', 'aggrLevel', 'isLeaf', 'customsCode', 'customsDesc', 'mosCode', 'motCode', 'motDesc', 'qtyUnitCode', 'qtyUnitAbbr', 'qty', 'isQtyEstimated', 'altQtyUnitCode', 'altQtyUnitAbbr', 'altQty', 'isAltQtyEstimated', 'netWgt', 'isNetWgtEstimated', 'grossWgt', 'isGrossWgtEstimated', 'cifvalue', 'fobvalue', 'primaryValue', 'legacyEstimationFlag', 'isReported', 'isAggregate']


,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
0,C,M,20241101,2024,11,202411,156,CHN,China,X,...,9212267.0,False,0.0,False,None,4655187.0,4655187.0,0,True,False
1,C,M,20241201,2024,12,202412,156,CHN,China,X,...,2933309.0,False,0.0,False,None,1534888.0,1534888.0,0,True,False


## 6. 전체 수집 (캐싱 + 체크포인트)

- 캐시 파일이 존재하면 API 호출 없이 로드한다.
- 할당량 소진(`None` 반환) 시 즉시 중단하고 진행 상황을 저장한다.
- 재실행하면 캐시에서 이어서 수집한다.

In [43]:
def normalize_response(
    df_raw: pd.DataFrame,
    reporter_code: str,
    partner_code: str,
    hs_code: str,
    start_yymm: str,
    end_yymm: str,
) -> pd.DataFrame:
    """
    Comtrade 응답 DataFrame을 파이프라인 표준 형식으로 정규화.

    출력 컬럼: trade_date, reporter_code, partner_code, hs_code_query,
              exp_dlr(수출금액), exp_qty(수출량)
    요청한 모든 월에 대해 행을 생성하고, 데이터가 없는 월은 0으로 채운다.
    """
    all_periods = [
        p.strftime("%Y%m")
        for p in pd.period_range(start_yymm, end_yymm, freq="M")
    ]

    if df_raw is None or df_raw.empty:
        return pd.DataFrame({
            "trade_date":    all_periods,
            "reporter_code": reporter_code,
            "partner_code":  partner_code,
            "hs_code_query": hs_code,
            "exp_dlr":       0,
            "exp_qty":       0,
        })

    df = df_raw.copy()

    # trade_date: refYear + refMonth 결합 또는 period 컬럼
    if "refYear" in df.columns and "refMonth" in df.columns:
        df["trade_date"] = (
            df["refYear"].astype(str).str.zfill(4)
            + df["refMonth"].astype(str).str.zfill(2)
        )
    elif "period" in df.columns:
        df["trade_date"] = df["period"].astype(str).str.replace(".", "", regex=False).str[:6]
    else:
        df["trade_date"] = "UNKNOWN"

    # 수출 금액·수량 컬럼 식별
    value_col    = next((c for c in ["primaryValue", "fobvalue", "cifvalue"] if c in df.columns), None)
    quantity_col = next((c for c in ["qty", "netWgt", "grossWgt", "primaryQuantity"] if c in df.columns), None)

    agg_dict = {}
    if value_col:
        df[value_col] = pd.to_numeric(df[value_col], errors="coerce").fillna(0)
        agg_dict["exp_dlr"] = (value_col, "sum")
    if quantity_col:
        df[quantity_col] = pd.to_numeric(df[quantity_col], errors="coerce").fillna(0)
        agg_dict["exp_qty"] = (quantity_col, "sum")

    if agg_dict:
        grouped = df.groupby("trade_date").agg(**agg_dict).reset_index()
    else:
        grouped = df[["trade_date"]].drop_duplicates().assign(exp_dlr=0, exp_qty=0)

    # 요청 기간 내 모든 월 보장 (missing월 = 0)
    period_frame = pd.DataFrame({"trade_date": all_periods})
    result = period_frame.merge(grouped, on="trade_date", how="left").fillna(0)
    result["reporter_code"] = reporter_code
    result["partner_code"]  = partner_code
    result["hs_code_query"] = hs_code

    for col in ["exp_dlr", "exp_qty"]:
        if col not in result.columns:
            result[col] = 0

    return result[["trade_date", "reporter_code", "partner_code", "hs_code_query", "exp_dlr", "exp_qty"]]

In [49]:
def cache_key(row: pd.Series, s: str, e: str) -> Path:
    """캐시 파일 경로 생성: {regulated_iso3}_{hs}_{partner_iso2}_{start}_{end}.parquet"""
    return CACHE_DIR / (
        f"{row['origin_country_iso3']}_{row['hs_code_query']}_"
        f"{row['country_iso2']}_{s}_{e}.parquet"
    )


results           = []
failed            = []
quota_hit         = False
consecutive_fails = 0   # 연속 실패 횟수: 3회 이상이면 할당량 소진으로 판단
total             = len(call_plan)

for i, row in call_plan.iterrows():
    if quota_hit:
        break

    chunks = make_monthly_chunks(row["window_start"], row["window_end"])
    chunk_results = []
    all_chunks_ok = True

    for s, e in chunks:
        cp = cache_key(row, s, e)

        if cp.exists():
            df_chunk = pd.read_parquet(cp)
            consecutive_fails = 0  # 캐시 로드 성공 시 초기화
        else:
            try:
                df_raw = collect_comtrade_one(
                    reporter_code=row["reporter_code"],
                    partner_code=row["partner_code"],
                    hs_code=str(row["hs_code_query"]),
                    start_yymm=s,
                    end_yymm=e,
                )
                df_chunk = normalize_response(
                    df_raw,
                    row["reporter_code"], row["partner_code"],
                    str(row["hs_code_query"]), s, e,
                )
                df_chunk.to_parquet(cp, index=False)
                consecutive_fails = 0
                time.sleep(SLEEP_BETWEEN_CALLS)

            except RuntimeError as exc:
                # 3회 재시도 포함 여전히 실패 → 연속 실패 카운트 증가
                consecutive_fails += 1
                print(f"  [{i+1}] 실패 ({consecutive_fails}회 연속): {exc}")

                if consecutive_fails >= 3:
                    # 3회 연속 실패 = 할당량 소진 또는 지속적 네트워크 오류
                    print("[수집 중단] 연속 3회 실패. 재실행 시 캐시에서 이어서 수집합니다.")
                    quota_hit = True
                    all_chunks_ok = False
                    break
                else:
                    # 단발성 오류 → 이 조합만 건너뛰고 계속
                    failed.append({
                        "idx": i,
                        "chunk": f"{s}~{e}",
                        "regulated": row["origin_country_iso3"],
                        "intermediary": row["country_iso2"],
                        "error": str(exc),
                    })
                    all_chunks_ok = False
                    break

            except Exception as exc:
                failed.append({
                    "idx": i,
                    "chunk": f"{s}~{e}",
                    "regulated": row["origin_country_iso3"],
                    "intermediary": row["country_iso2"],
                    "error": str(exc),
                })
                all_chunks_ok = False
                break

        chunk_results.append(df_chunk)

    if all_chunks_ok and chunk_results:
        df_event = pd.concat(chunk_results, ignore_index=True)
        df_event["origin_country_iso3"] = row["origin_country_iso3"]
        df_event["intermediary_iso2"]   = row["country_iso2"]
        df_event["window_start"]        = row["window_start"]
        df_event["window_end"]          = row["window_end"]
        results.append(df_event)

    if (i + 1) % CHECKPOINT_EVERY == 0 or i == total - 1 or quota_hit:
        print(f"[{i+1}/{total}] 완료={len(results)} / 실패={len(failed)} / 중단={quota_hit}")

print(f"\n수집 완료: {len(results)}건 성공, {len(failed)}건 실패")
if failed:
    print("실패 목록 (최대 5건):", failed[:5])

[20/735] 완료=20 / 실패=0 / 중단=False
[40/735] 완료=40 / 실패=0 / 중단=False
[60/735] 완료=60 / 실패=0 / 중단=False
[80/735] 완료=80 / 실패=0 / 중단=False
[100/735] 완료=100 / 실패=0 / 중단=False
[120/735] 완료=120 / 실패=0 / 중단=False
[140/735] 완료=140 / 실패=0 / 중단=False
[160/735] 완료=160 / 실패=0 / 중단=False
[180/735] 완료=180 / 실패=0 / 중단=False
[200/735] 완료=200 / 실패=0 / 중단=False
[220/735] 완료=220 / 실패=0 / 중단=False
[240/735] 완료=240 / 실패=0 / 중단=False
[260/735] 완료=260 / 실패=0 / 중단=False
[280/735] 완료=280 / 실패=0 / 중단=False
[300/735] 완료=300 / 실패=0 / 중단=False
[320/735] 완료=320 / 실패=0 / 중단=False
[340/735] 완료=340 / 실패=0 / 중단=False
[360/735] 완료=360 / 실패=0 / 중단=False
[380/735] 완료=380 / 실패=0 / 중단=False
[400/735] 완료=400 / 실패=0 / 중단=False
[420/735] 완료=420 / 실패=0 / 중단=False
[440/735] 완료=440 / 실패=0 / 중단=False
[460/735] 완료=460 / 실패=0 / 중단=False
[480/735] 완료=480 / 실패=0 / 중단=False
[500/735] 완료=500 / 실패=0 / 중단=False
[520/735] 완료=520 / 실패=0 / 중단=False
[540/735] 완료=540 / 실패=0 / 중단=False
[560/735] 완료=560 / 실패=0 / 중단=False
[580/735] 완료=580 / 실패=0 / 중단

## 7. 결과 병합 및 저장

In [50]:
if not results:
    raise RuntimeError("수집된 데이터가 없습니다. 위 수집 셀을 먼저 실행하세요.")

df_f1 = pd.concat(results, ignore_index=True)
print("수집 shape:", df_f1.shape)
df_f1.head()

수집 shape: (18375, 10)


,trade_date,reporter_code,partner_code,hs_code_query,exp_dlr,exp_qty,origin_country_iso3,intermediary_iso2,window_start,window_end
0,202411,156,392,720851,4655187.0,9212267.0,CHN,JP,2024-11,2026-11
1,202412,156,392,720851,1534888.0,2933309.0,CHN,JP,2024-11,2026-11
2,202501,156,392,720851,0.0,0.0,CHN,JP,2024-11,2026-11
3,202502,156,392,720851,0.0,0.0,CHN,JP,2024-11,2026-11
4,202503,156,392,720851,0.0,0.0,CHN,JP,2024-11,2026-11


In [51]:
# intermediary_candidates와 조인해 source_row_id, hs_code_query 등 메타 복원
merge_keys = ["origin_country_iso3", "hs_code_query", "window_start", "window_end", "intermediary_iso2"]

# intermediary_candidates의 country_iso2 → intermediary_iso2 rename
meta = intermediary_candidates.rename(columns={"country_iso2": "intermediary_iso2"})

# hs_code_query 타입 통일
df_f1["hs_code_query"] = df_f1["hs_code_query"].astype(str)
meta["hs_code_query"]  = meta["hs_code_query"].astype(str)

df_f1_final = df_f1.merge(meta, on=merge_keys, how="left")

print("최종 shape:", df_f1_final.shape)
print("\nNull counts:")
print(df_f1_final[["source_row_id", "trade_date", "exp_dlr"]].isnull().sum())

최종 shape: (18375, 14)

Null counts:
source_row_id    0
trade_date       0
exp_dlr          0
dtype: int64


df_f1_final.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"저장 완료: {OUTPUT_PATH}")
print(f"총 {len(df_f1_final)}행")